# 🔄 Step 3: Workflow Orchestration

## From Manual Handoffs to Automated Pipelines

In Step 2, we manually passed outputs between agents. Now we'll use MAF's `WorkflowBuilder`
to create an **automated incident response pipeline** with:

- **Sequential execution**: Triage → Diagnostics → Remediation → Verify → Comms
- **Conditional routing**: If verification fails → retry or escalate
- **Type-safe state**: Shared context flows through the pipeline
- **Single entry point**: One call handles the entire incident

---

## Architecture

```
┌──────────┐    ┌──────────────┐    ┌─────────────┐    ┌──────────┐    ┌───────┐
│  Triage  │───►│ Diagnostics  │───►│ Remediation │───►│  Verify  │───►│ Comms │
└──────────┘    └──────────────┘    └─────────────┘    └────┬─────┘    └───────┘
                                           ▲                │
                                           │    FAIL        │
                                           └────────────────┘
```

In [ ]:
import os
import sys
import json
from dataclasses import dataclass, field
sys.path.insert(0, "..")
from dotenv import load_dotenv

from agent_framework import WorkflowBuilder, WorkflowContext, executor
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import AzureCliCredential

from tools.mock_infra import (
    check_alert_history, get_runbook,
    get_metrics, get_logs, check_dependencies,
    restart_pod, scale_service, flush_cache, toggle_feature_flag, escalate_to_human,
    get_health_status, run_smoke_test,
    post_to_slack, create_incident_ticket, update_status_page,
)

load_dotenv("../.env")

# Load incident data
with open("../data/incidents.json") as f:
    incidents = json.load(f)

print("✅ Imports ready")

## Define Shared State

The workflow needs shared state to pass information between executors.
We use a `dataclass` to hold the incident context as it flows through the pipeline.

In [ ]:
@dataclass
class IncidentState:
    """Shared state that flows through the incident response workflow."""
    # Input
    alert_title: str = ""
    alert_service: str = ""
    alert_severity: str = ""
    alert_description: str = ""
    
    # Accumulated context from each stage
    triage_result: str = ""
    diagnostics_result: str = ""
    remediation_result: str = ""
    verification_result: str = ""
    comms_result: str = ""
    
    # Control flow
    is_resolved: bool = False
    retry_count: int = 0
    max_retries: int = 1

print("✅ IncidentState defined")

## Define Workflow Executors

Each executor wraps one of our specialized agents. The `@executor` decorator
tells MAF this is a step in the workflow.

### 🎯 YOUR TASK
Complete the `triage_executor` and `diagnostics_executor` below.
The remediation, verification, and comms executors are provided as reference.

In [ ]:
@executor
async def triage_executor(ctx: WorkflowContext[IncidentState]) -> str:
    """First step: Triage the incoming alert."""
    state = ctx.state
    print(f"\n{'='*60}")
    print(f"📋 TRIAGE STAGE")
    print(f"{'='*60}")
    
    # TODO: Create the triage agent with check_alert_history and get_runbook tools
    # TODO: Run it with the alert details from state
    # TODO: Store result in state.triage_result
    
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        agent = client.create_agent(
            name="TriageAgent",
            instructions="""You are an incident Triage Agent. When given an alert:
1. Use check_alert_history to check if this is a recurring issue
2. Use get_runbook to find the remediation playbook
3. Output: severity, recurring status, whether auto-remediation is allowed, recommended incident type for runbook lookup.
Be concise.""",
            tools=[check_alert_history, get_runbook],
        )
        
        result = await agent.run(
            f"Alert: {state.alert_title}\nService: {state.alert_service}\n"
            f"Severity: {state.alert_severity}\nDescription: {state.alert_description}"
        )
        
        state.triage_result = result.text
        print(f"\n{result.text}")
    
    return "diagnostics"  # Next step

In [ ]:
@executor
async def diagnostics_executor(ctx: WorkflowContext[IncidentState]) -> str:
    """Second step: Diagnose root cause."""
    state = ctx.state
    print(f"\n{'='*60}")
    print(f"🔍 DIAGNOSTICS STAGE")
    print(f"{'='*60}")
    
    # TODO: Create the diagnostics agent with get_metrics, get_logs, check_dependencies
    # TODO: Include triage context in the prompt
    # TODO: Store result in state.diagnostics_result
    
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        agent = client.create_agent(
            name="DiagnosticsAgent",
            instructions="""You are a Diagnostics Agent. Investigate the incident using your tools.
1. Use get_metrics to check resource utilization
2. Use get_logs to find error patterns
3. Use check_dependencies to identify cascading failures
Output: specific root cause with evidence, affected components, and exact remediation action needed.""",
            tools=[get_metrics, get_logs, check_dependencies],
        )
        
        result = await agent.run(
            f"Triage summary:\n{state.triage_result}\n\n"
            f"Service: {state.alert_service}\nInvestigate root cause."
        )
        
        state.diagnostics_result = result.text
        print(f"\n{result.text}")
    
    return "remediation"  # Next step

In [ ]:
@executor
async def remediation_executor(ctx: WorkflowContext[IncidentState]) -> str:
    """Third step: Execute the fix."""
    state = ctx.state
    print(f"\n{'='*60}")
    print(f"🔧 REMEDIATION STAGE (attempt {state.retry_count + 1})")
    print(f"{'='*60}")
    
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        agent = client.create_agent(
            name="RemediationAgent",
            instructions="""You are a Remediation Agent. Execute the fix based on diagnostics.
Use the appropriate tool based on the root cause:
- OOM/memory leak → restart_pod
- High CPU/traffic → scale_service
- Stale cache → flush_cache
- External dependency failure → toggle_feature_flag for failover
- Cannot auto-fix → escalate_to_human
Use exact names from the diagnostics (specific pod names, service names, flag names).""",
            tools=[restart_pod, scale_service, flush_cache, toggle_feature_flag, escalate_to_human],
        )
        
        result = await agent.run(
            f"Diagnostics findings:\n{state.diagnostics_result}\n\n"
            f"Service: {state.alert_service}\nExecute the appropriate fix."
        )
        
        state.remediation_result = result.text
        print(f"\n{result.text}")
    
    return "verification"  # Next step

In [ ]:
@executor
async def verification_executor(ctx: WorkflowContext[IncidentState]) -> str:
    """Fourth step: Verify the fix worked. Routes to comms or back to remediation."""
    state = ctx.state
    print(f"\n{'='*60}")
    print(f"✅ VERIFICATION STAGE")
    print(f"{'='*60}")
    
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        agent = client.create_agent(
            name="VerificationAgent",
            instructions="""You are a Verification Agent. Check if the fix worked.
1. Use get_health_status to check service health
2. Use run_smoke_test to run functional tests
End your response with exactly one of: VERDICT: RESOLVED or VERDICT: FAILED""",
            tools=[get_health_status, run_smoke_test],
        )
        
        result = await agent.run(
            f"Remediation performed:\n{state.remediation_result}\n\n"
            f"Service: {state.alert_service}\nVerify the fix worked."
        )
        
        state.verification_result = result.text
        print(f"\n{result.text}")
        
        # ROUTING LOGIC: Check if resolved or needs retry
        if "VERDICT: RESOLVED" in result.text.upper() or "RESOLVED" in result.text.upper():
            state.is_resolved = True
            return "communications"  # Success path
        else:
            state.retry_count += 1
            if state.retry_count >= state.max_retries:
                print("\n⚠️ Max retries reached — escalating to human")
                return "communications"  # Give up, notify team
            return "remediation"  # Retry the fix

In [ ]:
@executor
async def communications_executor(ctx: WorkflowContext[IncidentState]) -> None:
    """Final step: Notify team and create records."""
    state = ctx.state
    print(f"\n{'='*60}")
    print(f"📢 COMMUNICATIONS STAGE")
    print(f"{'='*60}")
    
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        agent = client.create_agent(
            name="CommunicationsAgent",
            instructions="""You are the Communications Agent. Notify the team about the incident resolution.
1. Post a concise summary to Slack (#incidents channel) with severity based on resolution status
2. Create an incident ticket with the full timeline
3. Update the status page to 'operational' if resolved, 'degraded' if not
Keep the Slack message brief (4-5 lines max).""",
            tools=[post_to_slack, create_incident_ticket, update_status_page],
        )
        
        status = "RESOLVED" if state.is_resolved else "ESCALATED (auto-fix failed)"
        
        result = await agent.run(
            f"Incident status: {status}\n\n"
            f"Timeline:\n- Triage: {state.triage_result[:200]}\n"
            f"- Diagnostics: {state.diagnostics_result[:200]}\n"
            f"- Remediation: {state.remediation_result[:200]}\n"
            f"- Verification: {state.verification_result[:200]}\n\n"
            f"Service: {state.alert_service}\n"
            f"Notify the team."
        )
        
        state.comms_result = result.text
        print(f"\n{result.text}")
    
    return None  # End of workflow

## Build and Run the Workflow

### 🎯 YOUR TASK
Wire the executors together using `WorkflowBuilder`. The flow is:

```
triage → diagnostics → remediation → verification → communications
                                 ▲         │
                                 └─ FAIL ──┘
```

In [ ]:
# Build the workflow
workflow = (
    WorkflowBuilder[IncidentState]()
    .add_executor(triage_executor, name="triage")
    .add_executor(diagnostics_executor, name="diagnostics")
    .add_executor(remediation_executor, name="remediation")
    .add_executor(verification_executor, name="verification")
    .add_executor(communications_executor, name="communications")
    .build()
)

print("✅ Workflow built!")
print("   Flow: triage → diagnostics → remediation → verification → communications")
print("   Retry: verification can loop back to remediation if fix fails")

In [ ]:
# Run the workflow on incident #1 (payment-api high latency)
alert = incidents[0]

print(f"🚨 INCIDENT RESPONSE INITIATED")
print(f"   Alert: {alert['title']}")
print(f"   Service: {alert['service']}")
print(f"   Severity: {alert['severity']}")
print(f"\n{'='*60}")

# Initialize state with the alert data
initial_state = IncidentState(
    alert_title=alert["title"],
    alert_service=alert["service"],
    alert_severity=alert["severity"],
    alert_description=alert["description"],
)

# Run the full pipeline
final_state = await workflow.run(state=initial_state)

print(f"\n\n{'='*60}")
print(f"🏁 INCIDENT RESPONSE COMPLETE")
print(f"{'='*60}")
print(f"   Resolved: {'✅ Yes' if final_state.is_resolved else '❌ No (escalated)'}")
print(f"   Retries: {final_state.retry_count}")

## 🧪 Try a Different Incident

Run the same workflow on incident #3 (notification-service email failures).
Notice how the agents make **different decisions** — toggle feature flag instead of restart.

In [ ]:
# Try incident #3: notification-service email failures
alert2 = incidents[2]

print(f"🚨 INCIDENT RESPONSE INITIATED")
print(f"   Alert: {alert2['title']}")
print(f"   Service: {alert2['service']}")
print(f"   Severity: {alert2['severity']}")

state2 = IncidentState(
    alert_title=alert2["title"],
    alert_service=alert2["service"],
    alert_severity=alert2["severity"],
    alert_description=alert2["description"],
)

final_state2 = await workflow.run(state=state2)

print(f"\n\n🏁 COMPLETE — Resolved: {'✅' if final_state2.is_resolved else '❌'}")

## 🎉 What We Achieved

With workflow orchestration, we now have:

| Feature | Before (Step 2) | After (Step 3) |
|---------|----------------|----------------|
| Agent coordination | Manual copy-paste | Automated pipeline |
| Error handling | None | Retry + escalation |
| Routing | Fixed sequence | Conditional (verify → retry or complete) |
| State management | Ad-hoc variables | Typed `IncidentState` dataclass |
| Reusability | One-off scripts | Same workflow handles any incident |

---

## ➡️ Next: Step 4 — Memory Patterns

The workflow works, but it doesn't **learn**. Every incident starts from scratch.
In Step 4, we'll add memory so the system:
- Remembers past incidents and their resolutions
- Gets faster at diagnosing recurring issues
- Builds institutional knowledge over time